# HW3 練功本 v2：看懂 `st_minecode_v2.ipynb` 多做了什麼

這本是**給我自己看的**，不用交。建議先做完 v1 的 `st_minecode_study.ipynb`，這本只講 v2 **新增或改掉**的部分。
和 v1 練功本一樣：只用幾行假資料，CPU 幾秒就跑完；每一節都是 **白話解說 → 🧪 示範 → ✍️ 手打練習 → 參考答案**。

練習格子跑的時候會自動對答案：⏳ = 還沒寫、✅ = 對了、❌ = 不對（會印出期望的結果）。

| 節 | 主題 | 對應 v2 notebook 的位置 | 建議 |
| :--- | :--- | :--- | :--- |
| 1 | 為什麼 v1 的證據會被截掉 | 第 4 節「v1 的截斷問題有多嚴重？」 | 看懂就好 |
| 2 | 組輸入：標準證據放前面 + 每句限長 | 第 3 節 `format_sentence` / `build_context` | ✍️ **手打** |
| 3 | hard negative：挑「很像但不是答案」的句子 | 第 3 節「微調 reranker」的資料 | ✍️ **手打** |
| 4 | listwise loss：8 選 1 的 cross entropy | 第 3 節 `finetune_reranker` | ✍️ **手打** |
| 5 | 凍結詞向量層 | 第 3 節 `finetune_reranker` 開頭 | ✍️ **手打** |
| 6 | 微調變差就改回來 | 第 3 節 `if ft_recall >= zs_recall` | 看懂就好 |
| 7 | gradient accumulation | 第 4 節 Training | ✍️ **手打** |
| 8 | 保留最好的 epoch（為什麼要 clone） | 第 4 節 Training 最後 | ✍️ **手打** |
| 9 | NEI bias ＋ reranker 門檻 | 第 5 節 `decide_labels` | ✍️ **手打** |
| 10 | large 模型為什麼要 batch 8 × 累積 2 | 第 1 節參數 | 看懂就好 |

In [ ]:
import random

import torch
import torch.nn.functional as F

torch.manual_seed(0)


def check(name, got, expected):
    """練習格子用的對答案小工具。"""
    if got is None or got is Ellipsis:
        print(f'⏳ {name}：還沒寫完（把 ... 換成你的程式）')
        return
    same = torch.allclose(got, expected, atol=1e-5) if isinstance(expected, torch.Tensor) else got == expected
    print(f'✅ {name}：正確！' if same else f'❌ {name}：不對\n   你的結果：{got}\n   期望結果：{expected}')

## 1. 為什麼 v1 的證據會被截掉？（看懂就好）

分類器的輸入是 `[CLS] claim [SEP] 5 句證據 [SEP]`，超過 `MAX_LENGTH` 的部分會被 `truncation='only_second'` **直接丟掉**（只丟證據，claim 保留）。
中文 BERT 幾乎是「一個字一個 token」，所以可以用字數粗估長度：

- claim 約 30 字；一句維基句子常常 40～80 字，再加上「【頁名】」。
- 5 句加起來很容易超過 256。

v1 訓練時把 5 句**打亂**，標準證據有可能被排到第 4、5 句，剛好落在被截掉的地方。
這時模型「看不到證據」，卻被要求回答 SUPPORTS / REFUTES，學到的就是雜訊。下面用假的長度模擬一下。

In [ ]:
# 🧪 示範：用「字數 ≈ token 數」模擬截斷
claim_len = 30
sentence_lens = [60, 75, 50, 90, 70]   # 5 句證據（含頁名）的字數，第 0 句是標準證據
special_tokens = 3                     # [CLS]、[SEP]、[SEP]


def gold_survives(order, max_length):
    """照 order 的順序排 5 句，標準證據（第 0 句）有沒有完整留在 max_length 之內。"""
    used = claim_len + special_tokens
    for index in order:
        used += sentence_lens[index]
        if index == 0:
            return used <= max_length
    return False


rng = random.Random(0)
trials = [rng.sample(range(5), 5) for _ in range(1000)]
print('v1（256、打亂）：標準證據留下來的比例 =', sum(gold_survives(order, 256) for order in trials) / len(trials))
print('v2（512、標準證據放第 1 句）：', gold_survives([0, 1, 2, 3, 4], 512))
print('就算還是 256，只要放第 1 句：', gold_survives([0, 1, 2, 3, 4], 256))

## 2. 組輸入：標準證據放前面 + 每句限長 ✍️

v2 的 `build_context` 做兩件事：
1. 每句前面加「【頁名】」，句子本身最多 `SENTENCE_CHAR_LIMIT` 字（字串切片 `sentence[:limit]`，超過的部分切掉，不夠長就整句保留）。
2. 把每句用空白接起來。順序由呼叫的人決定：訓練時是「標準證據 + 其他句」，測試時是 reranker 的排名。

Python 小提醒：`' '.join(list)` 把字串串列用空白接起來；`[f(x) for x in xs if f(x)]` 可以順便濾掉空字串。

In [ ]:
# ✍️ 手打練習：build_context(keys, sentences, limit)
# keys = [(頁名, 句號), ...]；sentences = {(頁名, 句號): 句子}；找不到的句子要略過
sentences = {('周杰倫', 0): '周杰倫是臺灣歌手', ('葉惠美', 1): '葉惠美是周杰倫在2003年發行的專輯，收錄晴天等歌曲',
             ('美國', 0): '美國位於北美洲'}


def build_context(keys, sentences, limit):
    parts = ...   # 每個 key → f'【{頁名}】{句子[:limit]}'；sentences.get(key) 找不到的略過
    return None   # 改成 ' '.join(parts)


check('build_context', build_context([('葉惠美', 1), ('周杰倫', 0), ('不存在', 9), ('美國', 0)], sentences, 10),
      '【葉惠美】葉惠美是周杰倫在20 【周杰倫】周杰倫是臺灣歌手 【美國】美國位於北美洲')

<details><summary>參考答案（先自己寫再點開）</summary>

```python
def build_context(keys, sentences, limit):
    parts = [f'【{page}】{sentences[(page, sid)][:limit]}' for page, sid in keys if sentences.get((page, sid))]
    return ' '.join(parts)
```
「葉惠美是周杰倫在2003年…」只留前 10 個字「葉惠美是周杰倫在20」。正式 notebook 的 `format_sentence` 是同一件事，只是用 `get_sentence` 去維基查句子。
</details>

## 3. hard negative：挑「很像但不是答案」的句子 ✍️

微調 reranker 時，每題要組一個 8 句的「考卷」：第 0 句是標準證據（正例），後面 7 句是負例。負例怎麼挑很重要：

| 負例來源 | 例子（claim：周杰倫在 2003 年發行葉惠美） | 對模型的幫助 |
| :--- | :--- | :--- |
| 隨便抽一句 | 「富士山是日本最高的山」 | 太簡單，模型一看就知道，學不到東西 |
| **hard negative**：向量初篩排很前面、但不是標準證據 | 「周杰倫是臺灣歌手」 | 主題對了但不能判斷真假，逼模型學會分辨 |

所以正式 notebook 從 bge-m3 初篩的前 14 名（排除所有標準證據句）裡隨機抽 7 句。**一定要先排除標準證據**，
不然同一句會同時是正例和負例，模型會被教亂。

In [ ]:
# ✍️ 手打練習：make_group(dense_keys, gold, n_neg)
# dense_keys：bge-m3 初篩的句子（已排好名次）；gold：標準證據句的 list
# 回傳：[正例] + 前 n_neg 個「不在 gold 裡」的句子（這裡不隨機，方便對答案）；負例不夠 n_neg 句就回傳空 list []
def make_group(dense_keys, gold, n_neg):
    negatives = ...   # dense_keys 裡不在 gold 的
    group = ...       # [gold[0]] + negatives 的前 n_neg 個
    return None       # 改成：負例不夠就回傳 []，否則回傳 group


dense_keys = [('周杰倫', 0), ('葉惠美', 1), ('周杰倫', 3), ('葉惠美', 0), ('晴天', 0), ('臺灣', 2)]
check('make_group', make_group(dense_keys, [('葉惠美', 1)], 3),
      [('葉惠美', 1), ('周杰倫', 0), ('周杰倫', 3), ('葉惠美', 0)])
check('負例不夠', make_group(dense_keys, [('葉惠美', 1)], 6), [])

<details><summary>參考答案</summary>

```python
def make_group(dense_keys, gold, n_neg):
    negatives = [key for key in dense_keys if key not in gold]
    group = [gold[0]] + negatives[:n_neg]
    return [] if len(negatives) < n_neg else group
```
第 2 個 check：去掉標準證據後只剩 5 句負例，不夠 6 句，所以回傳 `[]`。正式 notebook 遇到這種題目就直接跳過，不拿來微調。
正式 notebook 用 `rng.choice(gold)` 隨機挑正例、`rng.sample(negatives[:14], 7)` 隨機挑負例，每次看到的組合不同，比較不會死背。
</details>

## 4. listwise loss：8 選 1 的 cross entropy ✍️

reranker 一次吃進「claim + 1 句」，輸出 1 個分數。一組 8 句就會有 8 個分數，排成 `[題數, 8]`：

```text
           正例   負1   負2  ...  負7
第 1 題 [  3.2,  1.0,  2.9, ..., -0.5 ]   ← 希望第 0 個最大
第 2 題 [  0.1,  2.4,  0.3, ...,  0.0 ]   ← 這題 reranker 搞錯了，loss 會很大
```

把每一列當成「8 選 1 的分類題」，正確答案永遠是第 0 個，直接用 `F.cross_entropy(scores, target)`，`target` 全部是 0。
cross entropy = `-log(softmax(scores)[0])`：正例分數比其他 7 個高越多，softmax 給它的機率越接近 1，loss 越接近 0。

模型輸出的形狀是 `[題數 × 8, 1]`（每一對一個分數），要先 `.view(題數, 8)` 才能這樣算。

In [ ]:
# 🧪 示範：形狀怎麼變
flat_scores = torch.tensor([[3.2], [1.0], [2.9], [-0.5],    # 第 1 題的 4 句（這裡示範 1 正例 + 3 負例）
                            [0.1], [2.4], [0.3], [0.0]])     # 第 2 題
print('模型輸出：', tuple(flat_scores.shape), '→ view 之後：', tuple(flat_scores.view(2, 4).shape))
print(flat_scores.view(2, 4))

In [ ]:
# ✍️ 手打練習：listwise_loss(flat_scores, group_size)
def listwise_loss(flat_scores, group_size):
    scores = ...  # [題數 × group_size, 1] → [題數, group_size]
    target = ...  # 長度 = 題數，全部是 0（dtype=torch.long）
    return None   # 改成 F.cross_entropy(scores, target)


manual = -(torch.log_softmax(flat_scores.view(2, 4), dim=1)[:, 0]).mean()
check('listwise_loss', listwise_loss(flat_scores, 4), manual)

<details><summary>參考答案</summary>

```python
def listwise_loss(flat_scores, group_size):
    scores = flat_scores.view(-1, group_size)
    target = torch.zeros(scores.shape[0], dtype=torch.long)
    return F.cross_entropy(scores, target)
```
`view(-1, group_size)` 的 `-1` 是「這一維讓 PyTorch 自己算」。第 2 題的正例只有 0.1、比負例 2.4 低很多，所以它貢獻了大部分的 loss。
</details>

## 5. 凍結詞向量層 ✍️

`bge-reranker-v2-m3` 是 XLM-RoBERTa large：詞表 25 萬個詞 × 1024 維 ≈ **2.5 億個參數**都在詞向量（embedding）層。
訓練時，每個要更新的參數除了權重本身，還要存梯度和 Adam 的 2 個狀態，大約 **4 倍**的記憶體。
把詞向量層的 `requires_grad` 設成 `False`，它就不會被更新，也不用存那些東西，T4 才放得下。

在 Hugging Face 裡，不管是 BERT 還是 XLM-R，`model.base_model.embeddings` 都指向詞向量層。下面用一個超小的 BERT 練習。

In [ ]:
from transformers import BertConfig, BertForSequenceClassification

tiny = BertForSequenceClassification(BertConfig(vocab_size=1000, hidden_size=32, num_hidden_layers=2,
                                                num_attention_heads=2, intermediate_size=64, num_labels=1))
embedding_count = sum(p.numel() for p in tiny.base_model.embeddings.parameters())
print('全部參數：', sum(p.numel() for p in tiny.parameters()), '| 其中詞向量層：', embedding_count)

In [ ]:
# ✍️ 手打練習：freeze_embeddings(model) → 回傳「還可以訓練的參數數量」
def freeze_embeddings(model):
    embeddings = ...   # 詞向量層（提示：model.base_model.embeddings）
    ...                # 用 for 迴圈把 embeddings.parameters() 每一個的 requires_grad 設成 False
    return None        # 改成：requires_grad 還是 True 的參數，numel() 加總


check('freeze_embeddings', freeze_embeddings(tiny), sum(p.numel() for p in tiny.parameters()) - embedding_count)

<details><summary>參考答案</summary>

```python
def freeze_embeddings(model):
    embeddings = model.base_model.embeddings
    for parameter in embeddings.parameters():
        parameter.requires_grad = False
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
```
正式 notebook 之後把 `trainable = [p for p in model.parameters() if p.requires_grad]` 交給 AdamW，optimizer 就只管這些參數。
</details>

## 6. 微調變差就改回來（看懂就好）

微調不保證一定變好（學習率太大、資料太少都可能讓模型變差）。所以正式 notebook 的流程是：

```text
現成 reranker ──► 在 dev 上算 Recall（zs_recall）
      │
      ▼ 微調（只用 train）
微調後 reranker ──► 在 dev 上算 Recall（ft_recall）
      │
      ├─ ft_recall >= zs_recall ──► 用微調後的
      └─ 否則 ──────────────────► 重新載入現成的（load_reranker()）
```

這就是「用 dev 做 model selection」：dev 沒有拿來訓練，所以可以公平地比較兩個模型。test 永遠用 dev 上選出來的那個。

## 7. gradient accumulation ✍️

large 模型 + 512 token，T4 一次只放得下 8 筆。想要 batch size 16 的效果，可以：

1. 第 1 批 8 筆：算 loss → `backward()`（**先不 step**，梯度會存在 `.grad` 裡）
2. 第 2 批 8 筆：算 loss → `backward()`（PyTorch 的梯度是**累加**的，所以 `.grad` = 兩批的梯度相加）
3. 這時才 `optimizer.step()`，再 `zero_grad()`

因為 loss 是「平均」，兩批相加會變成 2 倍，所以每批的 loss 要先除以累積次數。這樣算出來的梯度，
會和「一次放 16 筆」**完全一樣**（下面的練習會驗證）。

In [ ]:
# 🧪 示範：PyTorch 的梯度是累加的
w = torch.tensor(1.0, requires_grad=True)
(w * 3).backward()
print('第 1 次 backward 後 w.grad =', w.grad.item())
(w * 3).backward()
print('第 2 次 backward 後 w.grad =', w.grad.item(), '← 累加了，所以平常每一步都要 zero_grad()')

In [ ]:
# ✍️ 手打練習：accumulated_grad(model, features, targets, chunks) → 回傳累積後的梯度
# 把 16 筆資料切成 chunks 份，每份算 loss、除以 chunks、backward；最後回傳 model.weight.grad 的複本
torch.manual_seed(0)
features, targets = torch.randn(16, 4), torch.randint(0, 3, (16,))
model = torch.nn.Linear(4, 3)


def accumulated_grad(model, features, targets, chunks):
    model.zero_grad()
    for x, y in zip(features.chunk(chunks), targets.chunk(chunks)):
        loss = ...   # F.cross_entropy(model(x), y) 再除以 chunks
        ...          # 反向傳播（不要 zero_grad，讓梯度累加）
    return None      # 改成 model.weight.grad.clone()


model.zero_grad()
F.cross_entropy(model(features), targets).backward()
full_batch_grad = model.weight.grad.clone()
check('累積 2 次 = 一次 16 筆', accumulated_grad(model, features, targets, 2), full_batch_grad)

<details><summary>參考答案</summary>

```python
def accumulated_grad(model, features, targets, chunks):
    model.zero_grad()
    for x, y in zip(features.chunk(chunks), targets.chunk(chunks)):
        loss = F.cross_entropy(model(x), y) / chunks
        loss.backward()
    return model.weight.grad.clone()
```
正式 notebook 的寫法：`scaler.scale(loss / GRAD_ACCUM_STEPS).backward()`，然後 `if step % GRAD_ACCUM_STEPS == 0` 才做裁剪、`step`、`zero_grad`。
`or step == len(train_loader)` 是為了最後剩下不足 2 批時也要更新。學習率排程的總步數也要改成「更新次數」= 批數 ÷ 2。
</details>

## 8. 保留最好的 epoch：為什麼要 `.clone()` ✍️

每個 epoch 結束都在 dev 上算 FEVER score，比之前好就存一份權重。這裡有個陷阱：
`model.state_dict()` 回傳的 tensor 和模型**共用同一塊記憶體**，模型繼續訓練，存下來的「最好權重」也會跟著變。
所以要 `.detach().cpu().clone()` 複製一份新的（`.cpu()` 順便搬到 RAM，不佔 GPU 記憶體）。

In [ ]:
# 🧪 示範：沒有 clone 的話會發生什麼事
toy = torch.nn.Linear(1, 1, bias=False)
torch.nn.init.constant_(toy.weight, 1.0)
saved_wrong = toy.state_dict()                                                  # 沒有複製
saved_right = {name: t.detach().clone() for name, t in toy.state_dict().items()}  # 複製一份
with torch.no_grad():
    toy.weight.add_(5.0)                                                        # 假裝繼續訓練，權重變了
print('模型現在的權重：', toy.weight.item())
print('沒 clone 存的「最好權重」：', saved_wrong['weight'].item(), '← 跟著變了！')
print('有 clone 存的：', saved_right['weight'].item())

In [ ]:
# ✍️ 手打練習：pick_best(fevers) → 回傳 (最好的 epoch 編號, 需要另外存權重的 epoch 編號 list)
# 規則：epoch 從 1 開始；分數「嚴格大於」目前最好才更新；最後一個 epoch 不用另外存（它就是現在模型的權重）
def pick_best(fevers):
    best_fever, best_epoch, saved = -1.0, 0, []
    for epoch, fever in enumerate(fevers, start=1):
        ...  # 比目前最好還好：更新 best_fever、best_epoch；如果不是最後一個 epoch，把 epoch 加進 saved
    return None  # 改成 (best_epoch, saved)


check('第 2 個 epoch 最好', pick_best([0.70, 0.74, 0.73]), (2, [1, 2]))
check('最後一個最好', pick_best([0.70, 0.72, 0.75]), (3, [1, 2]))

<details><summary>參考答案</summary>

```python
def pick_best(fevers):
    best_fever, best_epoch, saved = -1.0, 0, []
    for epoch, fever in enumerate(fevers, start=1):
        if fever > best_fever:
            best_fever, best_epoch = fever, epoch
            if epoch < len(fevers):
                saved.append(epoch)
    return (best_epoch, saved)
```
訓練完如果 `best_epoch != EPOCHS`，就 `classifier.load_state_dict(best_state)` 換回去。large 模型一份權重約 1.3 GB，所以只在需要時才存。
</details>

## 9. NEI bias ＋ reranker 門檻 ✍️

v1 只有一個旋鈕（NEI bias）。v2 多一個：reranker 給「第 1 名句子」的分數如果低於門檻，直接判 NEI。

為什麼合理？FEVER score 的規則是：SUPPORTS / REFUTES 的證據沒找到就是 0 分。連最像的句子分數都很低，
代表證據多半沒找到，這時猜 S/R 也拿不到分；改猜 NEI，萬一它真的是 NEI 題就賺到了。

門檻用「dev 上第 1 名分數的分位數」來設，例如 10% 分位數 = 分數最低的那 10% 題目改判 NEI。
兩個旋鈕的所有組合都在 dev 上試，挑 FEVER score 最高的（grid search）。

In [ ]:
# ✍️ 手打練習：decide_labels(labels, top_scores, threshold)
# labels：已經用 NEI bias 決定好的標籤；threshold 是 None 時不改；否則分數 < threshold 的改成 NEI
NEI = 'NOT ENOUGH INFO'


def decide_labels(labels, top_scores, threshold):
    if threshold is None:
        return labels
    return None  # 改成 list comprehension：分數 < threshold → NEI，否則保留原本的標籤


labels = ['SUPPORTS', 'REFUTES', 'SUPPORTS', NEI]
top_scores = [2.5, -1.2, 0.4, -3.0]
check('沒有門檻', decide_labels(labels, top_scores, None), labels)
check('門檻 = 0', decide_labels(labels, top_scores, 0.0), ['SUPPORTS', NEI, 'SUPPORTS', NEI])

<details><summary>參考答案</summary>

```python
def decide_labels(labels, top_scores, threshold):
    if threshold is None:
        return labels
    return [NEI if score < threshold else label for label, score in zip(labels, top_scores)]
```
正式 notebook 的 `decide_labels` 多做一步：先用 `logits_to_labels(logits, nei_bias)` 從 logit 算出標籤，再套門檻。
</details>

In [ ]:
# 🧪 示範：小小的 grid search（4 題 dev）
import numpy as np

gold_labels = ['SUPPORTS', NEI, 'REFUTES', NEI]
evidence_found = [True, False, False, False]   # 這 4 題的證據有沒有找到（NEI 題不用證據）
model_labels = ['SUPPORTS', 'SUPPORTS', 'REFUTES', NEI]
scores = [3.0, -0.8, -1.5, 0.2]


def toy_fever(predicted):
    return sum(p == g and (g == NEI or found) for p, g, found in zip(predicted, gold_labels, evidence_found)) / 4


for q in [0.0, 0.25, 0.5]:
    threshold = None if q == 0 else float(np.quantile(scores, q))
    # 這裡直接寫出門檻規則（和練習的答案一樣），所以就算上面練習還沒寫完，這格也能跑
    predicted = model_labels if threshold is None else [NEI if s < threshold else l for l, s in zip(model_labels, scores)]
    print(f'門檻分位數 {q:.0%}（{threshold}）→ {predicted} → FEVER = {toy_fever(predicted)}')
print('第 2 題（真的是 NEI、但分類器猜 SUPPORTS）分數很低，門檻把它救回來了；第 3 題證據沒找到，改判 NEI 也沒損失。')

## 10. large 模型為什麼要 batch 8 × 累積 2？（看懂就好）

粗略估算 fp16 混合精度訓練需要的 GPU 記憶體（不含 activation）：

| 項目 | 每個參數幾 bytes | base（102M） | large（326M） |
| :--- | :--- | :--- | :--- |
| fp32 權重 | 4 | 0.4 GB | 1.3 GB |
| 梯度 | 4 | 0.4 GB | 1.3 GB |
| Adam 的 2 個狀態 | 8 | 0.8 GB | 2.6 GB |
| 小計 | 16 | 1.6 GB | **5.2 GB** |

再加上 activation（每一層的中間結果，要留著給 backward 用），大約和「batch size × 長度 × 層數 × hidden size」成正比：
large 有 24 層、hidden 1024，長度又從 256 放寬到 512，同樣 batch 16 會超過 T4 的 16 GB。
所以一次放 8 筆，再累積 2 批（第 7 節），效果等於 batch 16。如果還是 `CUDA out of memory`，可以：

1. 把 `BATCH_SIZE` 改成 4、`GRAD_ACCUM_STEPS` 改成 4（等效 batch 一樣是 16）。
2. 或打開 `GRADIENT_CHECKPOINTING = True`：不存中間結果、backward 時重算，省很多記憶體，但慢約 30%。

In [ ]:
for name, params in [('base', 102e6), ('large', 326e6), ('reranker（凍結詞向量前）', 568e6), ('reranker（凍結後可訓練）', 310e6)]:
    print(f'{name:22s} 權重 + 梯度 + Adam ≈ {params * 16 / 1024 ** 3:.1f} GB')